Cloning the Repository (Experiment 4)
This block clones the specific branch exp4 of the repository from GitHub to the local environment.

Dataset Description:
Purpose: Clone the exp4 branch of the repository that contains additional scripts, datasets, or experiments related to the project.

In [1]:
!git clone --branch exp4 https://github.com/paulinaeb/IDaSec-project.git

Cloning into 'IDaSec-project'...
remote: Enumerating objects: 433, done.
remote: Counting objects: 100% (267/267), done.
remote: Compressing objects: 100% (211/211), done.
remote: Total 433 (delta 137), reused 162 (delta 48), pack-reused 166 (from 1)
Receiving objects: 100% (433/433), 41.69 MiB | 7.36 MiB/s, done.
Resolving deltas: 100% (197/197), done.
Updating files: 100% (57/57), done.


# **Necessary Imports**

In [2]:

import os
import numpy
import torch
import pickle
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from google.colab import drive
from transformers import DistilBertTokenizer, DistilBertModel
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score




# **Listing Dataset Files**

This block lists all **subfolders** and files in the **SMS dataset** folder, providing insights into the dataset structure and contents.


 Identify and list the available subfolders (**dataset splits**) and files within the dataset. This is important for understanding how the data is organized (e.g., **with variations like charswap, homoglyph, spacing, and mixed**).

In [3]:
# Path to the dataset folder
dataset_path = "IDaSec-project/dataset/sms"

# List all subfolders in dataset
subfolders = [f.name for f in os.scandir(dataset_path) if f.is_dir()]
print("available datasets:", subfolders)

sms_charswap_path = dataset_path + "/charswap"
sms_homoglyph_path = dataset_path + "/homoglyph"
sms_spacing_path = dataset_path + "/spacing"
sms_mixed_path = dataset_path + "/mixed"


sms_charswap_files = os.listdir(sms_charswap_path)
print(f"\nFiles in '{sms_charswap_path}': {sms_charswap_files}")

sms_mixed_files = os.listdir(sms_mixed_path)
print(f"\nFiles in '{sms_mixed_path}': {sms_mixed_files}")

sms_homoglyph_files = os.listdir(sms_homoglyph_path)
print(f"\nFiles in '{sms_homoglyph_path}': {sms_homoglyph_files}")

sms_spacing_files = os.listdir(sms_spacing_path)
print(f"\nFiles in '{sms_spacing_path}': {sms_spacing_files}")


available datasets: ['mixed', 'charswap', 'spacing', 'homoglyph']

Files in 'IDaSec-project/dataset/sms/charswap': ['train_with_charswap.csv', 'test_with_charswap.csv', 'val_with_charswap.csv']

Files in 'IDaSec-project/dataset/sms/mixed': ['train_mixed.csv', 'val_mixed.csv', 'test_mixed.csv']

Files in 'IDaSec-project/dataset/sms/homoglyph': ['train_homoglyph.csv', 'test_homoglyph.csv', 'val_homoglyph.csv']

Files in 'IDaSec-project/dataset/sms/spacing': ['val_spacing.csv', 'train_spacing.csv', 'test_spacing.csv']


# **Evasions**


The original SMS dataset (train, val, test) and its evasion variants (charswap, homoglyph, Mixed, spacing) from their respective subfolders.


**Original Dataset:** Contains spam and ham emails without any modifications.

**Evasion Datasets:**

**charswap:** Emails with character-swapping modifications.

**homoglyph:** Emails with homoglyph substitutions (e.g., replacing "o" with "0").

**Mixed:** Original Email column with 3 adversarial types (Your m33ting is sch3duled for tomorrow at 10 @M).

**spacing:** Emails with added or removed spaces.

# **Charswap Dataset**

In [4]:
sms_charswap_train = pd.read_csv(sms_charswap_path + "/test_with_charswap.csv")
sms_charswap_test = pd.read_csv(sms_charswap_path + "/train_with_charswap.csv")
sms_charswap_val = pd.read_csv(sms_charswap_path + "/val_with_charswap.csv")

sms_charswap_train.head()

,email,target,email_charswapped,was_augmented
0,"Oh right, ok. I'll make sure that i do loads o...",ham,"Oh right, ok. I'll make sure that i do loads o...",False
1,I am in tirupur. call you da.,ham,I am in tirupur. call you da.,False
2,No that just means you have a fat head,ham,No that just means you have a fat head,False
3,"You have won ?1,000 cash or a ?2,000 prize! To...",spam,"['You haev won? 1, 000 cahs or a? 2, 000 rpize...",True
4,Come aftr &lt;DECIMAL&gt; ..now i m cleaning t...,ham,Come aftr &lt;DECIMAL&gt; ..now i m cleaning t...,False


# **Homoglyph Dataset**

In [5]:
sms_homoglyph_train = pd.read_csv(sms_homoglyph_path + "/train_homoglyph.csv")
sms_homoglyph_test = pd.read_csv(sms_homoglyph_path + "/test_homoglyph.csv")
sms_homoglyph_val = pd.read_csv(sms_homoglyph_path + "/val_homoglyph.csv")

sms_homoglyph_train.head()

,email,target,email_homoglyph
0,What to think no one saying clearly. Ok leave ...,ham,What to think no one saying clearly. Ok leave ...
1,"FREE RING TONE just text \POLYS\"" to 87131. Th...",spam,"FREE R𝙸𝕹𝙶 T𝒪N𝙀 jʋs𝘵 tℯxt \𝙿𝙾LYS⧵"" 𝔱o 8𝟕131. Th..."
2,"Trust me. Even if isn't there, its there.",ham,"Trust me. Even if isn't there, its there."
3,Hi dear we saw dear. We both are happy. Where ...,ham,Hi dear we saw dear. We both are happy. Where ...
4,"URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...",spam,"URG𝐄NT, I𝑀PORTA𝕹T INFORMATION FOR O2 USER․ TOD..."


# **Mixed Dataset**

In [6]:
sms_mixed_train = pd.read_csv(sms_mixed_path + "/train_mixed.csv")
sms_mixed_test = pd.read_csv(sms_mixed_path + "/test_mixed.csv")
sms_mixed_val = pd.read_csv(sms_mixed_path + "/val_mixed.csv")

sms_mixed_train.head()

,original,adversarial_light,adversarial_medium,adversarial_heavy,target
0,What to think no one saying clearly. Ok leave ...,What to think no one saying clearly. Ok leave ...,What to think no one saying clearly. Ok leave ...,What to think no one saying clearly. Ok leave ...,ham
1,"FREE RING TONE just text \POLYS\"" to 87131. Th...","FREE RING TONE just text \POLYS\"" to 87131. Th...","FREE RING TONE just text \POLYS\"" to 87131. Th...","FREE RIGN ✅ lcient TONE ujst etxt \POLYS\"" to ...",spam
2,"Trust me. Even if isn't there, its there.","Trust me. Even if isn't there, its there.","Trust me. Even if isn't there, its there.","Trust me. Even if isn't there, its there.",ham
3,Hi dear we saw dear. We both are happy. Where ...,Hi dear we saw dear. We both are happy. Where ...,Hi dear we saw dear. We both are happy. Where ...,Hi dear we saw dear. We both are happy. Where ...,ham
4,"URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...","URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...","URGENT, 1MP0RT@NT INFORMATION F0R O2 USER. TOD...","URGNE,TI MPORTANT INFORAMTIO FNOR O2 U$3.R TOD...",spam


# **Spacing Dataset**

In [7]:
sms_spacing_train = pd.read_csv(sms_spacing_path + "/train_spacing.csv")
sms_spacing_test = pd.read_csv(sms_spacing_path + "/test_spacing.csv")
sms_spacing_val = pd.read_csv(sms_spacing_path + "/val_spacing.csv")

sms_spacing_train.head()

,email,target,email_spaced
0,What to think no one saying clearly. Ok leave ...,ham,What to think no one saying clearly. Ok leave ...
1,"FREE RING TONE just text \POLYS\"" to 87131. Th...",spam,"F REE RING TO NE j ust t e x t \P O L Y S\"" to..."
2,"Trust me. Even if isn't there, its there.",ham,"Trust me. Even if isn't there, its there."
3,Hi dear we saw dear. We both are happy. Where ...,ham,Hi dear we saw dear. We both are happy. Where ...
4,"URGENT, IMPORTANT INFORMATION FOR O2 USER. TOD...",spam,"U RGENT, IMPORT ANT IN FORMATION FOR O2 U S E ..."


# **Model Evaluation on Charswap Dataset**

These blocks evaluate the **fine-tuned model** on the datasets, computing **accuracy** and a detailed **classification report**. The dataset is tokenized and metadata is simulated for the evaluation process.


**Google Drive Mounting:**

Google Drive is mounted for access to the pre-trained model stored in the drive (**distilbert_finetuned.pt**).

Model Definition (**DistilBERTWithMetadata**):

The DistilBERTWithMetadata model class remains the same as previous, combining text processing with metadata for classification.


The charswap test dataset is loaded into a pandas DataFrame (**sms_charswap_test**).

Evaluation Function (evaluate_dataset):

**Text and Metadata:** The text_column (email texts) and label_column (target labels) are prepared. Metadata features like sender_score and time are simulated for evaluation.

**Batch Tokenization:** The input texts are tokenized in batches to reduce memory usage.

**Prediction:** The model is evaluated on the input batches, and predictions are stored.

**Metrics:** The accuracy and classification report are computed using sklearn.metrics.classification_report.

## **Evaluation Output**



**Accuracy:** The accuracy of the model on the test dataset is printed.

**Classification Report:** A detailed classification report is generated, showing precision, recall, and F1-score for both ham and spam classes.



In [13]:


# Set environment variable to handle memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Define the model class
class DistilBERTWithMetadata(nn.Module):
    def __init__(self, metadata_dim, dropout=0.3):
        super(DistilBERTWithMetadata, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.metadata_fc = nn.Linear(metadata_dim, 64)
        self.classifier = nn.Linear(768 + 64, 2)
        self.text_weight = 0.8
        self.metadata_weight = 0.2

    def forward(self, input_ids, attention_mask, metadata):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = bert_output.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)

        metadata_output = torch.relu(self.metadata_fc(metadata))
        metadata_output = self.dropout(metadata_output)

        weighted_text = self.text_weight * pooled_output
        weighted_metadata = self.metadata_weight * metadata_output
        combined = torch.cat((weighted_text, weighted_metadata), dim=-1)

        logits = self.classifier(combined)
        return logits

# Initialize model with correct metadata_dim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata_cols = ['sender_score', 'time']  # Match training metadata dimension (2)
model = DistilBERTWithMetadata(metadata_dim=len(metadata_cols)).to(device)
model.load_state_dict(torch.load('/content/drive/My Drive/distilbert_finetuned.pt'))
model.eval()

# Clear GPU memory
torch.cuda.empty_cache()

# Load the charswap test dataset
dataset_path = "IDaSec-project/dataset/sms"
sms_charswap_test = pd.read_csv(f"{dataset_path}/charswap/test_with_charswap.csv")

# Function to evaluate the dataset in batches
def evaluate_dataset(df, text_column, label_column='target', batch_size=16):
    # Prepare texts and labels
    texts = df[text_column].tolist()
    labels = df[label_column].map({'ham': 0, 'spam': 1}).values
    labels = torch.tensor(labels)

    # Simulate metadata (2 features: sender_score, time)
    metadata = torch.tensor(np.random.uniform(0, 1, (len(df), 2))).float()

    # Tokenize in batches
    all_input_ids = []
    all_attention_masks = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", max_length=512, padding="max_length", truncation=True)
        all_input_ids.append(inputs['input_ids'])
        all_attention_masks.append(inputs['attention_mask'])

    input_ids = torch.cat(all_input_ids, dim=0).to(device)
    attention_mask = torch.cat(all_attention_masks, dim=0).to(device)
    metadata = metadata.to(device)
    labels = labels.to(device)

    # Get predictions in batches
    model.eval()
    all_predicted = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch_input_ids = input_ids[i:i + batch_size]
            batch_attention_mask = attention_mask[i:i + batch_size]
            batch_metadata = metadata[i:i + batch_size]
            outputs = model(batch_input_ids, batch_attention_mask, batch_metadata)
            _, predicted = torch.max(outputs, dim=1)
            all_predicted.append(predicted)

    predicted = torch.cat(all_predicted, dim=0)

    # Compute metrics
    accuracy = (predicted == labels).float().mean().item()
    report = classification_report(labels.cpu(), predicted.cpu(), target_names=['ham', 'spam'])

    return accuracy, report, predicted, labels

# Evaluate the charswap test set
print("\nEvaluating Charswap Test Set:")
accuracy, report, predicted, labels = evaluate_dataset(sms_charswap_test, 'email_charswapped', batch_size=16)

# Print accuracy
print(f"Accuracy: {accuracy:.2f}")

# Print the detailed classification report in table format
print("\nDetailed Classification Report:")
print(report)



Mounted at /content/drive

Evaluating Charswap Test Set:
Accuracy: 0.97

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       966
        spam       0.98      0.83      0.89       149

    accuracy                           0.97      1115
   macro avg       0.97      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115



# **Model Evaluation on Homoglyph Dataset**

In [14]:

# Set environment variable to handle memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Define the model class
class DistilBERTWithMetadata(nn.Module):
    def __init__(self, metadata_dim, dropout=0.3):
        super(DistilBERTWithMetadata, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.metadata_fc = nn.Linear(metadata_dim, 64)
        self.classifier = nn.Linear(768 + 64, 2)
        self.text_weight = 0.8
        self.metadata_weight = 0.2

    def forward(self, input_ids, attention_mask, metadata):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = bert_output.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)

        metadata_output = torch.relu(self.metadata_fc(metadata))
        metadata_output = self.dropout(metadata_output)

        weighted_text = self.text_weight * pooled_output
        weighted_metadata = self.metadata_weight * metadata_output
        combined = torch.cat((weighted_text, weighted_metadata), dim=-1)

        logits = self.classifier(combined)
        return logits

# Initialize model with correct metadata_dim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata_cols = ['sender_score', 'time']  # Match training metadata dimension (2)
model = DistilBERTWithMetadata(metadata_dim=len(metadata_cols)).to(device)
model.load_state_dict(torch.load('/content/drive/My Drive/distilbert_finetuned.pt'))
model.eval()

# Clear GPU memory
torch.cuda.empty_cache()

# Load the homoglyph test dataset
dataset_path = "IDaSec-project/dataset/sms"
sms_homoglyph_test = pd.read_csv(f"{dataset_path}/homoglyph/test_homoglyph.csv")

# Function to evaluate the dataset in batches
def evaluate_dataset(df, text_column, label_column='target', batch_size=16):
    # Prepare texts and labels
    texts = df[text_column].tolist()
    labels = df[label_column].map({'ham': 0, 'spam': 1}).values
    labels = torch.tensor(labels)

    # Simulate metadata (2 features: sender_score, time)
    metadata = torch.tensor(np.random.uniform(0, 1, (len(df), 2))).float()

    # Tokenize in batches
    all_input_ids = []
    all_attention_masks = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", max_length=512, padding="max_length", truncation=True)
        all_input_ids.append(inputs['input_ids'])
        all_attention_masks.append(inputs['attention_mask'])

    input_ids = torch.cat(all_input_ids, dim=0).to(device)
    attention_mask = torch.cat(all_attention_masks, dim=0).to(device)
    metadata = metadata.to(device)
    labels = labels.to(device)

    # Get predictions in batches
    model.eval()
    all_predicted = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch_input_ids = input_ids[i:i + batch_size]
            batch_attention_mask = attention_mask[i:i + batch_size]
            batch_metadata = metadata[i:i + batch_size]
            outputs = model(batch_input_ids, batch_attention_mask, batch_metadata)
            _, predicted = torch.max(outputs, dim=1)
            all_predicted.append(predicted)

    predicted = torch.cat(all_predicted, dim=0)

    # Compute metrics
    accuracy = (predicted == labels).float().mean().item()
    report = classification_report(labels.cpu(), predicted.cpu(), target_names=['ham', 'spam'])

    return accuracy, report, predicted, labels

# Evaluate the homoglyph test set
print("\nEvaluating Homoglyph Test Set:")
accuracy, report, predicted, labels = evaluate_dataset(sms_homoglyph_test, 'email_homoglyph', batch_size=16)

# Print accuracy
print(f"Accuracy: {accuracy:.2f}")

# Print the detailed classification report in table format
print("\nDetailed Classification Report:")
print(report)



Evaluating Homoglyph Test Set:
Accuracy: 0.88

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       0.88      1.00      0.93       966
        spam       0.83      0.10      0.18       149

    accuracy                           0.88      1115
   macro avg       0.86      0.55      0.56      1115
weighted avg       0.87      0.88      0.83      1115

Text: Oh right, ok. I'll make sure that i do loads of wo...
True Label: 0 (0 = ham, 1 = spam)
Predicted Label: 0 (0 = ham, 1 = spam)
---
Text: I am in tirupur. call you da....
True Label: 0 (0 = ham, 1 = spam)
Predicted Label: 0 (0 = ham, 1 = spam)
---
Text: No that just means you have a fat head...
True Label: 0 (0 = ham, 1 = spam)
Predicted Label: 0 (0 = ham, 1 = spam)
---


# **Model Evaluation on Mixed Dataset (Adversarial Heavy)**


In [15]:

# Set environment variable to handle memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Define the model class
class DistilBERTWithMetadata(nn.Module):
    def __init__(self, metadata_dim, dropout=0.3):
        super(DistilBERTWithMetadata, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.metadata_fc = nn.Linear(metadata_dim, 64)
        self.classifier = nn.Linear(768 + 64, 2)
        self.text_weight = 0.8
        self.metadata_weight = 0.2

    def forward(self, input_ids, attention_mask, metadata):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = bert_output.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)

        metadata_output = torch.relu(self.metadata_fc(metadata))
        metadata_output = self.dropout(metadata_output)

        weighted_text = self.text_weight * pooled_output
        weighted_metadata = self.metadata_weight * metadata_output
        combined = torch.cat((weighted_text, weighted_metadata), dim=-1)

        logits = self.classifier(combined)
        return logits

# Initialize model with correct metadata_dim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata_cols = ['sender_score', 'time']  # Match training metadata dimension (2)
model = DistilBERTWithMetadata(metadata_dim=len(metadata_cols)).to(device)
model.load_state_dict(torch.load('/content/drive/My Drive/distilbert_finetuned.pt'))
model.eval()

# Clear GPU memory
torch.cuda.empty_cache()

# Load the mixed test dataset
dataset_path = "IDaSec-project/dataset/sms"
sms_mixed_test = pd.read_csv(f"{dataset_path}/mixed/test_mixed.csv")

# Function to evaluate the dataset in batches
def evaluate_dataset(df, text_column, label_column='target', batch_size=16):
    # Prepare texts and labels
    texts = df[text_column].tolist()
    labels = df[label_column].map({'ham': 0, 'spam': 1}).values
    labels = torch.tensor(labels)

    # Simulate metadata (2 features: sender_score, time)
    metadata = torch.tensor(np.random.uniform(0, 1, (len(df), 2))).float()

    # Tokenize in batches
    all_input_ids = []
    all_attention_masks = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", max_length=512, padding="max_length", truncation=True)
        all_input_ids.append(inputs['input_ids'])
        all_attention_masks.append(inputs['attention_mask'])

    input_ids = torch.cat(all_input_ids, dim=0).to(device)
    attention_mask = torch.cat(all_attention_masks, dim=0).to(device)
    metadata = metadata.to(device)
    labels = labels.to(device)

    # Get predictions in batches
    model.eval()
    all_predicted = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch_input_ids = input_ids[i:i + batch_size]
            batch_attention_mask = attention_mask[i:i + batch_size]
            batch_metadata = metadata[i:i + batch_size]
            outputs = model(batch_input_ids, batch_attention_mask, batch_metadata)
            _, predicted = torch.max(outputs, dim=1)
            all_predicted.append(predicted)

    predicted = torch.cat(all_predicted, dim=0)

    # Compute metrics
    accuracy = (predicted == labels).float().mean().item()
    report = classification_report(labels.cpu(), predicted.cpu(), target_names=['ham', 'spam'])

    return accuracy, report, predicted, labels

# Evaluate the mixed test set (using adversarial_heavy)
print("\nEvaluating Mixed Test Set (Adversarial Heavy):")
accuracy, report, predicted, labels = evaluate_dataset(sms_mixed_test, 'adversarial_heavy', batch_size=16)

# Print accuracy
print(f"Accuracy: {accuracy:.2f}")

# Print the detailed classification report in table format
print("\nDetailed Classification Report:")
print(report)



Evaluating Mixed Test Set (Adversarial Heavy):
Accuracy: 0.99

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00       966
        spam       0.98      0.98      0.98       149

    accuracy                           0.99      1115
   macro avg       0.99      0.99      0.99      1115
weighted avg       0.99      0.99      0.99      1115

Text: Oh right, ok. I'll make sure that i do loads of wo...
True Label: 0 (0 = ham, 1 = spam)
Predicted Label: 0 (0 = ham, 1 = spam)
---
Text: I am in tirupur. call you da....
True Label: 0 (0 = ham, 1 = spam)
Predicted Label: 0 (0 = ham, 1 = spam)
---
Text: No that just means you have a fat head...
True Label: 0 (0 = ham, 1 = spam)
Predicted Label: 0 (0 = ham, 1 = spam)
---


# **Model Evaluation on Mixed Dataset (Adversarial Medium)**

In [18]:

# Set environment variable to handle memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Define the model class
class DistilBERTWithMetadata(nn.Module):
    def __init__(self, metadata_dim, dropout=0.3):
        super(DistilBERTWithMetadata, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.metadata_fc = nn.Linear(metadata_dim, 64)
        self.classifier = nn.Linear(768 + 64, 2)
        self.text_weight = 0.8
        self.metadata_weight = 0.2

    def forward(self, input_ids, attention_mask, metadata):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = bert_output.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)

        metadata_output = torch.relu(self.metadata_fc(metadata))
        metadata_output = self.dropout(metadata_output)

        weighted_text = self.text_weight * pooled_output
        weighted_metadata = self.metadata_weight * metadata_output
        combined = torch.cat((weighted_text, weighted_metadata), dim=-1)

        logits = self.classifier(combined)
        return logits

# Initialize model with correct metadata_dim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata_cols = ['sender_score', 'time']  # Match training metadata dimension (2)
model = DistilBERTWithMetadata(metadata_dim=len(metadata_cols)).to(device)
model.load_state_dict(torch.load('/content/drive/My Drive/distilbert_finetuned.pt'))
model.eval()

# Clear GPU memory
torch.cuda.empty_cache()

# Load the mixed test dataset
dataset_path = "IDaSec-project/dataset/sms"
sms_mixed_test = pd.read_csv(f"{dataset_path}/mixed/test_mixed.csv")

# Function to evaluate the dataset in batches
def evaluate_dataset(df, text_column, label_column='target', batch_size=16):
    # Prepare texts and labels
    texts = df[text_column].tolist()
    labels = df[label_column].map({'ham': 0, 'spam': 1}).values
    labels = torch.tensor(labels)

    # Simulate metadata (2 features: sender_score, time)
    metadata = torch.tensor(np.random.uniform(0, 1, (len(df), 2))).float()

    # Tokenize in batches
    all_input_ids = []
    all_attention_masks = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", max_length=512, padding="max_length", truncation=True)
        all_input_ids.append(inputs['input_ids'])
        all_attention_masks.append(inputs['attention_mask'])

    input_ids = torch.cat(all_input_ids, dim=0).to(device)
    attention_mask = torch.cat(all_attention_masks, dim=0).to(device)
    metadata = metadata.to(device)
    labels = labels.to(device)

    # Get predictions in batches
    model.eval()
    all_predicted = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch_input_ids = input_ids[i:i + batch_size]
            batch_attention_mask = attention_mask[i:i + batch_size]
            batch_metadata = metadata[i:i + batch_size]
            outputs = model(batch_input_ids, batch_attention_mask, batch_metadata)
            _, predicted = torch.max(outputs, dim=1)
            all_predicted.append(predicted)

    predicted = torch.cat(all_predicted, dim=0)

    # Compute metrics
    accuracy = (predicted == labels).float().mean().item()
    report = classification_report(labels.cpu(), predicted.cpu(), target_names=['ham', 'spam'])

    return accuracy, report, predicted, labels

# Evaluate the mixed test set (using adversarial_medium)
print("\nEvaluating Mixed Test Set (Adversarial Medium):")
accuracy, report, predicted, labels = evaluate_dataset(sms_mixed_test, 'adversarial_medium', batch_size=16)

# Print accuracy
print(f"Accuracy: {accuracy:.2f}")

# Print the detailed classification report in table format
print("\nDetailed Classification Report:")
print(report)



Evaluating Mixed Test Set (Adversarial Medium):
Accuracy: 0.99

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00       966
        spam       0.98      0.98      0.98       149

    accuracy                           0.99      1115
   macro avg       0.99      0.99      0.99      1115
weighted avg       0.99      0.99      0.99      1115



# **Model Evaluation on Mixed Dataset (Adversarial Medium)**

In [20]:

# Set environment variable to handle memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Define the model class
class DistilBERTWithMetadata(nn.Module):
    def __init__(self, metadata_dim, dropout=0.3):
        super(DistilBERTWithMetadata, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(dropout)
        self.metadata_fc = nn.Linear(metadata_dim, 64)
        self.classifier = nn.Linear(768 + 64, 2)
        self.text_weight = 0.8
        self.metadata_weight = 0.2

    def forward(self, input_ids, attention_mask, metadata):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = bert_output.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)

        metadata_output = torch.relu(self.metadata_fc(metadata))
        metadata_output = self.dropout(metadata_output)

        weighted_text = self.text_weight * pooled_output
        weighted_metadata = self.metadata_weight * metadata_output
        combined = torch.cat((weighted_text, weighted_metadata), dim=-1)

        logits = self.classifier(combined)
        return logits

# Initialize model with correct metadata_dim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
metadata_cols = ['sender_score', 'time']  # Match training metadata dimension (2)
model = DistilBERTWithMetadata(metadata_dim=len(metadata_cols)).to(device)
model.load_state_dict(torch.load('/content/drive/My Drive/distilbert_finetuned.pt'))
model.eval()

# Clear GPU memory
torch.cuda.empty_cache()

# Load the spacing test dataset
dataset_path = "IDaSec-project/dataset/sms"
sms_spacing_test = pd.read_csv(f"{dataset_path}/spacing/test_spacing.csv")

# Function to evaluate the dataset in batches
def evaluate_dataset(df, text_column, label_column='target', batch_size=16):
    # Prepare texts and labels
    texts = df[text_column].tolist()
    labels = df[label_column].map({'ham': 0, 'spam': 1}).values
    labels = torch.tensor(labels)

    # Simulate metadata (2 features: sender_score, time)
    metadata = torch.tensor(np.random.uniform(0, 1, (len(df), 2))).float()

    # Tokenize in batches
    all_input_ids = []
    all_attention_masks = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, return_tensors="pt", max_length=512, padding="max_length", truncation=True)
        all_input_ids.append(inputs['input_ids'])
        all_attention_masks.append(inputs['attention_mask'])

    input_ids = torch.cat(all_input_ids, dim=0).to(device)
    attention_mask = torch.cat(all_attention_masks, dim=0).to(device)
    metadata = metadata.to(device)
    labels = labels.to(device)

    # Get predictions in batches
    model.eval()
    all_predicted = []
    with torch.no_grad():
        for i in range(0, len(df), batch_size):
            batch_input_ids = input_ids[i:i + batch_size]
            batch_attention_mask = attention_mask[i:i + batch_size]
            batch_metadata = metadata[i:i + batch_size]
            outputs = model(batch_input_ids, batch_attention_mask, batch_metadata)
            _, predicted = torch.max(outputs, dim=1)
            all_predicted.append(predicted)

    predicted = torch.cat(all_predicted, dim=0)

    # Compute metrics
    accuracy = (predicted == labels).float().mean().item()
    report = classification_report(labels.cpu(), predicted.cpu(), target_names=['ham', 'spam'])

    return accuracy, report, predicted, labels

# Evaluate the spacing test set
print("\nEvaluating Spacing Test Set:")
accuracy, report, predicted, labels = evaluate_dataset(sms_spacing_test, 'email_spaced', batch_size=16)

# Print accuracy
print(f"Accuracy: {accuracy:.2f}")

# Print the detailed classification report in table format
print("\nDetailed Classification Report:")
print(report)



fatal: destination path 'IDaSec-project' already exists and is not an empty directory.
Mounted at /content/drive

Evaluating Spacing Test Set:
Accuracy: 0.97

Detailed Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       0.97      0.79      0.87       149

    accuracy                           0.97      1115
   macro avg       0.97      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115

